In [ ]:
!pip install opencv-python ultralytics gdown --quiet
!pip install -q gdown

import cv2 ##baca videonya##
import gdown ##ambil dri gdrive##
from ultralytics import YOLO
import numpy as np
import csv
from google.colab.patches import cv2_imshow
import os
import time

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.8 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:

#data video ke gdrive
VIDEOS = {
    "1": {
        "label": "Video malam",
        "file_id": "1ZRDOnrA3XPgSX0hyKwRySNPtE7JzmyrW",
        "path": "video_malam.mp4",
    },
    "2": {
        "label": "Video siang (mesin beroperasi)",
        "file_id": "1gAx__Pik7C2yPbj1i-6LQbAkI3xFirj5",
        "path": "video_siang_operasi.mp4",
    },
    "3": {
        "label": "Video mesin dibuka",
        "file_id": "1izfx1u4N4PvFDP4-jHxaWIMsFqIK9JPW",  # ← ID BARU
        "path": "video_mesin_buka.mp4",
    },
    "4": {
        "label": "Video mesin ditutup",
        "file_id": "1t2ug2VGI4jYA2mL6HAvi726a-Qgo510u",
        "path": "video_mesin_tutup.mp4",
    },
}

#bikin piliha input
print("=== PILIH VIDEO YANG AKAN DIPROSES ===")
for key, data in VIDEOS.items():
    print(f"{key}. {data['label']}")

choice = input("Masukkan pilihan (1-4): ").strip()

if choice not in VIDEOS:
    raise ValueError("❌ Pilihan tidak valid. Harus 1–4.")

selected = VIDEOS[choice]
file_id = selected["file_id"]
video_path = selected["path"]

print("\n=== VIDEO DIPILIH ===")
print("Label   :", selected["label"])
print("File ID :", file_id)
print("Path    :", video_path)

#downloadnya
def download_drive(file_id, out_path, retries=2):

    # hapus file kosong
    if os.path.exists(out_path) and os.path.getsize(out_path) == 0:
        os.remove(out_path)

    # skip kalau sudah valid
    if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        return True

    url = f"https://drive.google.com/uc?export=download&id={file_id}"

    for attempt in range(retries):
        print(f"\nDownloading... attempt {attempt+1}/{retries}")
        gdown.download(url, out_path, quiet=False, fuzzy=True)

        if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
            return True

        if os.path.exists(out_path):
            os.remove(out_path)

        time.sleep(1)

    return False

#download kalo belum ada
ok = download_drive(file_id, video_path, retries=2)

#validasi file
if ok:
    print("\n✅ Video siap dipakai.")
    print("Exists :", True)
    print("Size   :", os.path.getsize(video_path), "bytes")
else:
    raise RuntimeError(
        "❌ Download gagal atau file kosong.\n"
        "Pastikan Google Drive di-set ke: Anyone with the link → Viewer\n"
        "Atau kemungkinan Drive kena quota limit."
    )

print("\nvideo_path sudah terset dan siap digunakan:", video_path)


=== PILIH VIDEO YANG AKAN DIPROSES ===
1. Video malam
2. Video siang (mesin beroperasi)
3. Video mesin dibuka
4. Video mesin ditutup
Masukkan pilihan (1-4): 1

=== VIDEO DIPILIH ===
Label   : Video malam
File ID : 1ZRDOnrA3XPgSX0hyKwRySNPtE7JzmyrW
Path    : video_malam.mp4

Downloading... attempt 1/2


Downloading...
From (original): https://drive.google.com/uc?id=1ZRDOnrA3XPgSX0hyKwRySNPtE7JzmyrW
From (redirected): https://drive.google.com/uc?id=1ZRDOnrA3XPgSX0hyKwRySNPtE7JzmyrW&confirm=t&uuid=9837767a-36ee-48d4-8fba-aec53413970c
To: /content/video_malam.mp4
100%|██████████| 836M/836M [00:11<00:00, 75.5MB/s]


✅ Video siap dipakai.
Exists : True
Size   : 835695324 bytes

video_path sudah terset dan siap digunakan: video_malam.mp4


fix Code

In [ ]:
print("video_path =", video_path)

#roi
PANEL_X, PANEL_Y, PANEL_W, PANEL_H = 370, 653, 40, 20
ROLL_X,  ROLL_Y,  ROLL_W,  ROLL_H  = 600, 455, 340, 140
ENV_X, ENV_Y, ENV_W, ENV_H = 50, 35, 80, 35
ENV_THR = 80

#trasehol
NORMAL_DELTA_THR = 15
NORMAL_STD_CAP   = 70
NORMAL_MIN_PANEL = 75

DARK_DELTA_THR = 14
DARK_STD_CAP   = 70
DARK_MIN_PANEL = 55

AROUND_M = 20
PANEL_ON_N  = 5
PANEL_OFF_N = 5

PANEL_OFF_T_DAY = 160  # <= OFF
PANEL_ON_T_DAY  = 190  # >= ON
#ini bikin biar ga naik turun ga jelas

#fd roller
PIX_THR = 12
GAUSS_K = (5, 5) #untuk gaussian blurnya
SKIP_N  = 1
OPEN_ITERS   = 1
DILATE_ITERS = 2
EMA_ALPHA = 0.35
#untuk menentukan off/idle
ON_THR  = 280
OFF_THR = 120
ON_N  = 3
OFF_N = 6

#biar ON ga spike ga jelas, ini biar pergerakan di luar mesin harusnya ha kebaca
ON_SEC = 2

#ignore set up
PERSON_CLASS_ID = 0
PERSON_CONF_IGNORE = 0.35
IGNORE_PERSON_ON_ROLL  = True #biat ngeignore objek yang mirip org
IGNORE_PERSON_ON_PANEL = True

ROLL_OCCLUDE_RATIO  = 0.12 #biar kalo ketutup berapa angka yang di butuhin agar bisa ngedetect ini
PANEL_OCCLUDE_RATIO = 0.20

# YOLO operator (hanya saat ON final)
OP_CONF = 0.35

#yolo
YOLO_WEIGHTS = "yolov8n.pt" #maasukin yolonya
RUN_YOLO_IGNORE_EVERY_N_FRAMES = 5 #dinyalain 5 frame sekali

#bikin log
LOG_PER_SECOND = True
csv_path = "/content/log_status.csv"

#utility (ini soal roinya)
def clamp_roi(x, y, rw, rh, W, H):
    x = max(0, min(int(x), W - 1))
    y = max(0, min(int(y), H - 1))
    rw = max(1, min(int(rw), W - x))
    rh = max(1, min(int(rh), H - y))
    return x, y, rw, rh

def safe_crop(gray, x, y, rw, rh):
    H, W = gray.shape
    x1 = max(0, min(int(x), W-1))
    y1 = max(0, min(int(y), H-1))
    x2 = max(0, min(int(x + rw), W))
    y2 = max(0, min(int(y + rh), H))
    return gray[y1:y2, x1:x2]

def overlap_ratio_roi(roi_x1, roi_y1, roi_x2, roi_y2, bx1, by1, bx2, by2):
    ix1 = max(roi_x1, bx1)
    iy1 = max(roi_y1, by1)
    ix2 = min(roi_x2, bx2)
    iy2 = min(roi_y2, by2)
    iw = max(0, ix2 - ix1)
    ih = max(0, iy2 - iy1)
    inter = iw * ih
    if inter <= 0:
        return 0.0
    roi_area = max(1, (roi_x2 - roi_x1) * (roi_y2 - roi_y1))
    return inter / float(roi_area)

def person_occludes_roi(yolo_result, roi_x, roi_y, roi_w, roi_h,
                        person_cls=0, conf_thr=0.35, min_ratio=0.15):
    if yolo_result is None or yolo_result.boxes is None:
        return False
    rx1, ry1, rx2, ry2 = roi_x, roi_y, roi_x + roi_w, roi_y + roi_h
    for box in yolo_result.boxes:
        cls = int(box.cls[0])
        conf = float(box.conf[0])
        if cls == person_cls and conf >= conf_thr:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            r = overlap_ratio_roi(rx1, ry1, rx2, ry2, x1, y1, x2, y2)
            if r >= min_ratio:
                return True
    return False

#ngebuka videonya
cap = cv2.VideoCapture(video_path)
assert cap.isOpened(), "Video tidak kebuka. Pastikan file ada & path benar."
#ngambil frame dan data dari videonya
fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
if not fps or fps <= 1: #kalo fps nya ga kebacay pake 25
    fps = 25.0
frames_per_sec = int(round(fps))

#ini biar roinya tetap di aman terhadap ukuran video
PANEL_X, PANEL_Y, PANEL_W, PANEL_H = clamp_roi(PANEL_X, PANEL_Y, PANEL_W, PANEL_H, w, h)
ROLL_X,  ROLL_Y,  ROLL_W,  ROLL_H  = clamp_roi(ROLL_X,  ROLL_Y,  ROLL_W,  ROLL_H,  w, h)
ENV_X,   ENV_Y,   ENV_W,   ENV_H   = clamp_roi(ENV_X, ENV_Y, ENV_W, ENV_H, w, h)

#manggil yolo lagi utnuk cek operator
model = YOLO(YOLO_WEIGHTS)

#state awal (kernel morfolofi, buffer frame roller, ukuran buffer)
kernel = np.ones((3, 3), np.uint8)
roll_buffer = []
buffer_max = max(2, SKIP_N + 1)

#variable untuk gerakan, status sementasa, counter on off
movement = 0
movement_ema = 0.0
status_hold = "IDLE"
on_cnt = 0
off_cnt = 0

#status mesin dari sisi panel
panel_on_count = 0
panel_off_count = 0
mesin_aktif = False

#keputusan untuk terang gelap
LAMP_ON_N  = 3
LAMP_OFF_N = 3 #***
lamp_on_count = 0
lamp_off_count = 0
lampu_terang = True

#untuk pecatatan frame dan penomoran
frame_idx = 0
last_logged_second = -1
on_sec_count = 0

#simpan hasil frame sementara, biar ga inteferensi tiap frame
yolo_cache = None
yolo_cache_frame = -999999

#untuk membuka file csv, menulis header
csv_file = open(csv_path, "w", newline="", encoding="utf-8")
csv_writer = csv.writer(csv_file)
csv_writer.writerow([
    "frame_idx","second",
    "lampu","env_mean",
    "panel_mean","panel_max","around_mean","delta","std_panel","panel_valid",
    "panel_occluded",
    "mesin_aktif_panel",
    "roll_occluded","operator_near_roll",
    "movement","movement_ema","status_raw","on_cnt","off_cnt",
    "on_sec_count","status_final",
    "operator_status"
])

def should_log_this_row(second_now): #untuk mesukin log perdetiknya
    global last_logged_second
    if LOG_PER_SECOND:
        if second_now == last_logged_second:
            return False
        last_logged_second = second_now
        return True
    return True

#ini untuk ngulang loop prosesnya selama video
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_idx += 1
    detik_now = frame_idx // frames_per_sec

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) #frame nya di konfersi ke gray scale

    #ini detect terang gelap nya
    env_gray = safe_crop(gray, ENV_X, ENV_Y, ENV_W, ENV_H)
    env_mean = float(np.mean(env_gray))
    terang_raw = (env_mean >= ENV_THR)

    if terang_raw:
        lamp_on_count += 1
        lamp_off_count = 0
    else:
        lamp_off_count += 1
        lamp_on_count = 0

    if lamp_on_count >= LAMP_ON_N:
        lampu_terang = True
    elif lamp_off_count >= LAMP_OFF_N:
        lampu_terang = False

    status_lampu = "TERANG" if lampu_terang else "GELAP"

    if lampu_terang:
        DELTA_THR = NORMAL_DELTA_THR
        STD_CAP   = NORMAL_STD_CAP
        MIN_PANEL_MEAN = NORMAL_MIN_PANEL
    else:
        DELTA_THR = DARK_DELTA_THR
        STD_CAP   = DARK_STD_CAP
        MIN_PANEL_MEAN = DARK_MIN_PANEL

    # 0.5) YOLO cache untuk occlusion / ignore (tidak tiap frame)
    need_cache = (IGNORE_PERSON_ON_ROLL or IGNORE_PERSON_ON_PANEL)
    if need_cache and (frame_idx - yolo_cache_frame) >= RUN_YOLO_IGNORE_EVERY_N_FRAMES:
        yolo_cache = model(frame, verbose=False)[0]
        yolo_cache_frame = frame_idx

    # biar panel dan semua roi itu ga salah detect kalo ketutup orang
    panel_occluded = False
    roll_occluded  = False
    if need_cache and (yolo_cache is not None):
        if IGNORE_PERSON_ON_PANEL:
            panel_occluded = person_occludes_roi(
                yolo_cache, PANEL_X, PANEL_Y, PANEL_W, PANEL_H,
                person_cls=PERSON_CLASS_ID,
                conf_thr=PERSON_CONF_IGNORE,
                min_ratio=PANEL_OCCLUDE_RATIO
            )
        if IGNORE_PERSON_ON_ROLL:
            roll_occluded = person_occludes_roi(
                yolo_cache, ROLL_X, ROLL_Y, ROLL_W, ROLL_H,
                person_cls=PERSON_CLASS_ID,
                conf_thr=PERSON_CONF_IGNORE,
                min_ratio=ROLL_OCCLUDE_RATIO
            )

    #ini on off panelnya
    mean_panel = 0.0
    max_panel  = 0
    mean_around = 0.0
    delta = 0.0
    std_panel = 0.0
    panel_valid = None  # None = HOLD

    if not panel_occluded:
        panel_gray = safe_crop(gray, PANEL_X, PANEL_Y, PANEL_W, PANEL_H)
        ax = PANEL_X - AROUND_M
        ay = PANEL_Y - AROUND_M
        aw = PANEL_W + 2 * AROUND_M
        ah = PANEL_H + 2 * AROUND_M
        around_gray = safe_crop(gray, ax, ay, aw, ah)

        mean_panel  = float(np.mean(panel_gray))
        max_panel   = int(np.max(panel_gray))
        mean_around = float(np.mean(around_gray))
        delta = mean_panel - mean_around
        std_panel = float(np.std(panel_gray))

        if lampu_terang:
            # DAY
            if mean_panel <= PANEL_OFF_T_DAY:
                panel_valid = False
            elif mean_panel >= PANEL_ON_T_DAY:
                panel_valid = True
            else:
                panel_valid = None  # HOLD
        else:
            # malam
            panel_valid = (delta >= DELTA_THR) and (std_panel <= STD_CAP) and (mean_panel >= MIN_PANEL_MEAN)

        #ini nentuin mana yang on mana yang off
        if panel_valid is True:
            panel_on_count += 1
            panel_off_count = 0
        elif panel_valid is False:
            panel_off_count += 1
            panel_on_count = 0
        else:
            pass  # HOLD

        if panel_on_count >= PANEL_ON_N:
            mesin_aktif = True
        elif panel_off_count >= PANEL_OFF_N:
            mesin_aktif = False
    # kalau panel_occluded: HOLD -> mesin_aktif tidak diubah

    #ini untuk status pergerakan rollnya
    operator_near_roll = False
    status_mesin_raw = "OFF"#kalo mesinaktif true baru masu ke sini, kalo ngga ya off aja
    operator_status = "TIDAK ADA"

    if not mesin_aktif:
        status_mesin_raw = "OFF"
        status_hold = "IDLE"
        movement = 0
        movement_ema = 0.0
        on_cnt = 0
        off_cnt = 0
        roll_buffer.clear()
        on_sec_count = 0
    else:
        operator_near_roll = bool(roll_occluded)
        #ambil roi roller (grays skale, gaussian blur)
        roll = frame[ROLL_Y:ROLL_Y + ROLL_H, ROLL_X:ROLL_X + ROLL_W]
        roll_gray = cv2.cvtColor(roll, cv2.COLOR_BGR2GRAY)
        roll_gray = cv2.GaussianBlur(roll_gray, GAUSS_K, 0)

        roll_buffer.append(roll_gray)
        if len(roll_buffer) > buffer_max:
            roll_buffer.pop(0)

        status_mesin_raw = status_hold
      #fd nya di lakukan, bendingin frame sekarang sama yang sebelumnya
        ref = None
        if len(roll_buffer) >= (SKIP_N + 1):
            ref = roll_buffer[-(SKIP_N + 1)]
        elif len(roll_buffer) >= 2:
            ref = roll_buffer[-2]

        #cari oerbedaan pixle nya
        if ref is not None:
            diff = cv2.absdiff(roll_gray, ref)
            _, th = cv2.threshold(diff, PIX_THR, 255, cv2.THRESH_BINARY)
            if OPEN_ITERS > 0:
                th = cv2.morphologyEx(th, cv2.MORPH_OPEN, kernel, iterations=OPEN_ITERS) #bersihin npise dan memperjelas area gerak
            th = cv2.dilate(th, kernel, iterations=DILATE_ITERS)

            #ingitung pixle yang bedanya (EMA)
            movement = int(cv2.countNonZero(th))
            movement_ema = EMA_ALPHA * movement + (1 - EMA_ALPHA) * movement_ema

            if operator_near_roll:
                status_mesin_raw = "IDLE"
                on_cnt = 0
                off_cnt = min(OFF_N, off_cnt + 1)
            else:
                if movement_ema >= ON_THR:  ##keputusan on idle (panel aktif + roll diam = idle )
                    on_cnt = min(ON_N, on_cnt + 1)
                    off_cnt = 0
                elif movement_ema <= OFF_THR:
                    off_cnt = min(OFF_N, off_cnt + 1)
                    on_cnt = 0

                if on_cnt >= ON_N: #keputusan on (panel aktif + roller aktif=on)
                    status_mesin_raw = "ON"
                elif off_cnt >= OFF_N:
                    status_mesin_raw = "IDLE"

#ini kontinu on biar dia ga spike aneh
        status_hold = status_mesin_raw
    if status_mesin_raw == "ON" and mesin_aktif and (not operator_near_roll):
        on_sec_count += 1
    else:
        on_sec_count = 0
#status finalnya
    if not mesin_aktif:
        status_mesin_final = "OFF"
    else:
        status_mesin_final = "ON" if on_sec_count >= ON_SEC else "IDLE"

    #nyalain ketika udh final on
    if status_mesin_final == "ON":
        yolo_op = model(frame, verbose=False)[0]
        operator_status = "TIDAK ADA"
        if yolo_op.boxes is not None:
            for box in yolo_op.boxes:
                cls = int(box.cls[0])
                conf = float(box.conf[0])
                if cls == PERSON_CLASS_ID and conf >= OP_CONF: #kalo ke detect ada operator
                    operator_status = "ADA"
                    break

    # LOG + PRINT
    if should_log_this_row(detik_now):
        flag = "IGNORE_SETUP" if operator_near_roll else "OK"
        panel_valid_out = -1 if panel_valid is None else int(panel_valid)

        csv_writer.writerow([
            frame_idx, detik_now,
            status_lampu, f"{env_mean:.2f}",
            f"{mean_panel:.2f}", int(max_panel), f"{mean_around:.2f}", f"{delta:.2f}", f"{std_panel:.2f}", panel_valid_out,
            int(panel_occluded),
            int(mesin_aktif),
            int(roll_occluded), int(operator_near_roll),
            int(movement), f"{movement_ema:.2f}", status_mesin_raw, int(on_cnt), int(off_cnt),
            int(on_sec_count), status_mesin_final,
            operator_status
        ])

        if status_mesin_final == "ON":
            print(f"Detik {detik_now}: {status_lampu}, Mesin {status_mesin_final}, Operator {operator_status}, EMA {int(movement_ema)}, {flag}")
        else:
            print(f"Detik {detik_now}: {status_lampu}, Mesin {status_mesin_final}, EMA {int(movement_ema)}, {flag}")

# CLEANUP
cap.release()
csv_file.close()

print("\nSELESAI.")
print("Output CSV:", csv_path)


video_path = video_malam.mp4
Detik 0: TERANG, Mesin OFF, EMA 0, OK
Detik 1: GELAP, Mesin OFF, EMA 0, OK
Detik 2: GELAP, Mesin OFF, EMA 0, OK
Detik 3: GELAP, Mesin OFF, EMA 0, OK
Detik 4: GELAP, Mesin OFF, EMA 0, OK
Detik 5: GELAP, Mesin OFF, EMA 0, OK
Detik 6: GELAP, Mesin OFF, EMA 0, OK
Detik 7: GELAP, Mesin OFF, EMA 0, OK
Detik 8: GELAP, Mesin OFF, EMA 0, OK
Detik 9: GELAP, Mesin OFF, EMA 0, OK
Detik 10: GELAP, Mesin OFF, EMA 0, OK
Detik 11: GELAP, Mesin OFF, EMA 0, OK
Detik 12: GELAP, Mesin OFF, EMA 0, OK
Detik 13: GELAP, Mesin OFF, EMA 0, OK
Detik 14: GELAP, Mesin OFF, EMA 0, OK
Detik 15: GELAP, Mesin OFF, EMA 0, OK
Detik 16: GELAP, Mesin OFF, EMA 0, OK
Detik 17: GELAP, Mesin OFF, EMA 0, OK
Detik 18: GELAP, Mesin OFF, EMA 0, OK
Detik 19: GELAP, Mesin OFF, EMA 0, OK
Detik 20: GELAP, Mesin OFF, EMA 0, OK
Detik 21: GELAP, Mesin OFF, EMA 0, OK
Detik 22: GELAP, Mesin OFF, EMA 0, OK
Detik 23: GELAP, Mesin OFF, EMA 0, OK
Detik 24: GELAP, Mesin OFF, EMA 0, OK
Detik 25: GELAP, Mesin OFF, EM

In [ ]:
from google.colab import files
files.download(csv_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

cek kalibrasi

nyibain trrasehold